# 10

In [38]:
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split

covtype = fetch_covtype()

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    covtype.data, covtype.target, random_state=42
)

In [43]:
X_train.data.shape

(435759, 54)

In [28]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

mlp_clf = MLPClassifier(hidden_layer_sizes=[100, 100, 100], early_stopping=True,
                        verbose=True, random_state=42)
pipeline = make_pipeline(StandardScaler(), mlp_clf)
pipeline.fit(X_train, y_train)

Iteration 1, loss = 0.56620846
Validation score: 0.797870
Iteration 2, loss = 0.43539499
Validation score: 0.828323
Iteration 3, loss = 0.38018900
Validation score: 0.853314
Iteration 4, loss = 0.34512917
Validation score: 0.863296
Iteration 5, loss = 0.31965615
Validation score: 0.872636
Iteration 6, loss = 0.30171112
Validation score: 0.877295
Iteration 7, loss = 0.28662288
Validation score: 0.883307
Iteration 8, loss = 0.27596983
Validation score: 0.888517
Iteration 9, loss = 0.26411366
Validation score: 0.890146
Iteration 10, loss = 0.25557785
Validation score: 0.893083
Iteration 11, loss = 0.24815619
Validation score: 0.898316
Iteration 12, loss = 0.24179882
Validation score: 0.898637
Iteration 13, loss = 0.23527023
Validation score: 0.901551
Iteration 14, loss = 0.22882145
Validation score: 0.901666
Iteration 15, loss = 0.22431561
Validation score: 0.905980
Iteration 16, loss = 0.22056695
Validation score: 0.906715
Iteration 17, loss = 0.21544139
Validation score: 0.901437
Iterat

,steps,"[('standardscaler', ...), ('mlpclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,hidden_layer_sizes,"[100, 100, ...]"
,activation,'relu'
,solver,'adam'
,alpha,0.0001


In [41]:
mlp_clf.best_validation_score_

0.9300302919038003

In [42]:
pipeline.score(X_test, y_test)

0.928841400866075

We tested with hidden layers (100, 100, 100) and it could achieve 92.88% accuracy on test set (not >93%). We need to do hyperparameter tuning. Let's consider 300 total neurons.

In [59]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform
from scipy.stats import loguniform

pipeline_new = make_pipeline(
    StandardScaler(), 
    MLPClassifier(early_stopping=True, verbose=True, random_state=42))

param_dist = {
    "mlpclassifier__hidden_layer_sizes": [(200, 100), (150, 150), (150, 100, 50), (100, 100, 100)],
    "mlpclassifier__alpha": uniform(0.0001, 0.01),
    "mlpclassifier__learning_rate_init": loguniform(0.0001, 0.01),
    "mlpclassifier__activation": ["relu", "tanh"],
}

search = RandomizedSearchCV(
    pipeline_new,
    param_distributions=param_dist,
    n_iter=32,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=True
)

X_small, _, y_small, _ = train_test_split(
    X_train,
    y_train,
    train_size=100000,
    stratify=y_train,
    random_state=42
)

search.fit(X_small, y_small)

Fitting 3 folds for each of 32 candidates, totalling 96 fits
Iteration 1, loss = 0.72650840
Validation score: 0.725400
Iteration 2, loss = 0.60566974
Validation score: 0.741300
Iteration 3, loss = 0.57234690
Validation score: 0.756900
Iteration 4, loss = 0.54407785
Validation score: 0.758000
Iteration 5, loss = 0.52068690
Validation score: 0.771500
Iteration 6, loss = 0.50014038
Validation score: 0.777700
Iteration 7, loss = 0.48268296
Validation score: 0.783900
Iteration 8, loss = 0.46658917
Validation score: 0.797300
Iteration 9, loss = 0.45057716
Validation score: 0.801600
Iteration 10, loss = 0.43596575
Validation score: 0.806700
Iteration 11, loss = 0.42323387
Validation score: 0.817600
Iteration 12, loss = 0.41053931
Validation score: 0.820200
Iteration 13, loss = 0.39850641
Validation score: 0.826500
Iteration 14, loss = 0.38676978
Validation score: 0.837900
Iteration 15, loss = 0.37668189
Validation score: 0.840500
Iteration 16, loss = 0.36775182
Validation score: 0.840300
Iter

,estimator,Pipeline(step...rbose=True))])
,param_distributions,"{'mlpclassifier__activation': ['relu', 'tanh'], 'mlpclassifier__alpha': <scipy.stats....0014548C39790>, 'mlpclassifier__hidden_layer_sizes': [(200, ...), (150, ...), ...], 'mlpclassifier__learning_rate_init': <scipy.stats....001454CA8BB70>}"
,n_iter,32
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,True
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [65]:
import pandas as pd

results = pd.DataFrame(search.cv_results_)
top3 = results.sort_values(by="mean_test_score", ascending=False).head(3)

for i, row in top3.iterrows():
    print(f"{i}:")
    print("Accuracy:", row["mean_test_score"])
    print("Params:", row["params"])
    print()

7:
Accuracy: 0.9043100123807383
Params: {'mlpclassifier__activation': 'tanh', 'mlpclassifier__alpha': np.float64(0.0005666566321361543), 'mlpclassifier__hidden_layer_sizes': (100, 100, 100), 'mlpclassifier__learning_rate_init': np.float64(0.0005404103854647331)}

21:
Accuracy: 0.9024800195803002
Params: {'mlpclassifier__activation': 'tanh', 'mlpclassifier__alpha': np.float64(0.00975255307264138), 'mlpclassifier__hidden_layer_sizes': (100, 100, 100), 'mlpclassifier__learning_rate_init': np.float64(0.001217284708112243)}

15:
Accuracy: 0.9013600017802542
Params: {'mlpclassifier__activation': 'tanh', 'mlpclassifier__alpha': np.float64(0.007653614103176525), 'mlpclassifier__hidden_layer_sizes': (150, 150), 'mlpclassifier__learning_rate_init': np.float64(0.001096821720752952)}



In [62]:
search.best_params_

{'mlpclassifier__activation': 'tanh',
 'mlpclassifier__alpha': np.float64(0.0005666566321361543),
 'mlpclassifier__hidden_layer_sizes': (100, 100, 100),
 'mlpclassifier__learning_rate_init': np.float64(0.0005404103854647331)}

Now, using the best params to make the final best model and train it on the entire train set.

In [64]:
best_model = make_pipeline(
    StandardScaler(), 
    MLPClassifier(
        activation='tanh', 
        alpha=0.000566, 
        learning_rate_init=0.000540,
        hidden_layer_sizes=(100, 100, 100), 
        early_stopping=True, 
        verbose=True, 
        random_state=42
    ),
)

best_model.fit(X_train, y_train)

Iteration 1, loss = 0.60799764
Validation score: 0.767120
Iteration 2, loss = 0.49155015
Validation score: 0.800532
Iteration 3, loss = 0.43027657
Validation score: 0.831352
Iteration 4, loss = 0.38488284
Validation score: 0.849780
Iteration 5, loss = 0.35051137
Validation score: 0.862378
Iteration 6, loss = 0.32444745
Validation score: 0.871443
Iteration 7, loss = 0.30332176
Validation score: 0.880714
Iteration 8, loss = 0.28587992
Validation score: 0.886704
Iteration 9, loss = 0.27075077
Validation score: 0.892303
Iteration 10, loss = 0.25806859
Validation score: 0.895699
Iteration 11, loss = 0.24651124
Validation score: 0.901299
Iteration 12, loss = 0.23671892
Validation score: 0.902997
Iteration 13, loss = 0.22756380
Validation score: 0.908138
Iteration 14, loss = 0.21936855
Validation score: 0.909446
Iteration 15, loss = 0.21194312
Validation score: 0.911281
Iteration 16, loss = 0.20499230
Validation score: 0.916215
Iteration 17, loss = 0.19896874
Validation score: 0.914999
Iterat

,steps,"[('standardscaler', ...), ('mlpclassifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,hidden_layer_sizes,"(100, ...)"
,activation,'tanh'
,solver,'adam'
,alpha,0.000566


In [66]:
best_model.named_steps['mlpclassifier'].best_validation_score_

0.9519919221589866

In [67]:
best_model.score(X_test, y_test)

0.9508719269137298

Achieved 95% accuracy on test set which is pretty good I guess.